# ARIA v4.0 - Hualien Disaster Accessibility Assessment

**Complete Integration and Analysis Notebook**

## Project Overview
This notebook implements the full ARIA v4.0 system for disaster accessibility assessment in Hualien County, integrating:
- Road network analysis (OSMnx + NetworkX)
- Dynamic rainfall-based weighting
- Pre/post-disaster isochrone analysis
- Accessibility impact assessment

**Submission Files Generated:**
- `hualien_network.graphml` (Road network data)
- `accessibility_benefit_cost_table.csv` (Analysis results)

In [ ]:
# Environment Setup
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from shapely.geometry import Point, MultiPoint, Polygon
import warnings
import json
from datetime import datetime
warnings.filterwarnings('ignore')

# Configuration
rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False

# Project settings
PROJECT_CRS = 'EPSG:3826'
NETWORK_DIST = 5000  # meters
TARGET_LOCATION = "Hualien City, Taiwan"
NETWORK_TYPE = 'drive'

print(f"ARIA v4.0 - Hualien Disaster Accessibility Assessment")
print(f"Target: {TARGET_LOCATION}")
print(f"CRS: {PROJECT_CRS}")
print(f"Network radius: {NETWORK_DIST}m")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Step 1: Road Network Extraction
print(f"?? Extracting road network for {TARGET_LOCATION}...")

# Increase timeout for reliability
ox.settings.timeout = 300
ox.settings.use_cache = True

try:
    # Extract road network
    G = ox.graph_from_address(TARGET_LOCATION, dist=NETWORK_DIST, network_type=NETWORK_TYPE)
    print(f"?? Extraction successful: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    # Project to meter coordinates
    G_proj = ox.project_graph(G, to_crs=PROJECT_CRS)
    print(f"?? Projection complete: {G_proj.graph['crs']}")
    
    # Calculate network bounds
    node_coords = [(G_proj.nodes[node]['x'], G_proj.nodes[node]['y']) for node in G_proj.nodes()]
    min_x, min_y = min(coord[0] for coord in node_coords), min(coord[1] for coord in node_coords)
    max_x, max_y = max(coord[0] for coord in node_coords), max(coord[1] for coord in node_coords)
    network_extent_x = max_x - min_x
    network_extent_y = max_y - min_y
    
    print(f"Network extent: {network_extent_x:.1f}m × {network_extent_y:.1f}m")
    print(f"Network area: ~{network_extent_x * network_extent_y / 1e6:.2f} km²")
    
except Exception as e:
    print(f"?? Extraction failed: {e}")
    print("Using fallback: NTU campus network for demonstration")
    
    # Fallback to NTU for demonstration
    G = ox.graph_from_address("National Taiwan University, Taipei", dist=1500, network_type='drive')
    G_proj = ox.project_graph(G, to_crs=PROJECT_CRS)
    print(f"Fallback network: {G_proj.number_of_nodes()} nodes, {G_proj.number_of_edges()} edges")

In [ ]:
# Step 2: Travel Time Calculation
print("?? Calculating base travel times...")

# Highway speed defaults (km/h)
speed_defaults = {
    'motorway': 110, 'motorway_link': 80,
    'trunk': 100, 'trunk_link': 60,
    'primary': 80, 'primary_link': 50,
    'secondary': 60, 'secondary_link': 40,
    'tertiary': 50, 'tertiary_link': 30,
    'residential': 40, 'living_street': 10,
    'unclassified': 30,
}

def get_speed_kph(data):
    """Get speed from OSM edge data"""
    maxspeed = data.get('maxspeed', None)
    if maxspeed:
        try:
            return float(maxspeed)
        except (ValueError, TypeError):
            if isinstance(maxspeed, list):
                try:
                    return float(maxspeed[0])
                except:
                    pass
    highway = data.get('highway', 'residential')
    if isinstance(highway, list):
        highway = highway[0]
    return speed_defaults.get(highway, 40)

# Calculate travel times for all edges
for u, v, k, data in G_proj.edges(data=True, keys=True):
    length = data['length']  # meters
    speed_kph = get_speed_kph(data)
    speed_ms = speed_kph / 3.6  # Convert to m/s
    data['travel_time_normal'] = length / speed_ms  # seconds
    data['speed_kph'] = speed_kph

# Statistics
travel_times = [data['travel_time_normal'] for _, _, _, data in G_proj.edges(data=True, keys=True)]
print(f"?? Travel time calculation complete")
print(f"Average travel time: {np.mean(travel_times):.1f}s")
print(f"Travel time range: {np.min(travel_times):.1f}s - {np.max(travel_times):.1f}s")

In [ ]:
# Step 3: Betweenness Centrality Analysis
print("?? Calculating betweenness centrality...")

# Calculate betweenness centrality using edge length as weight
centrality = nx.betweenness_centrality(G_proj, weight='length')

# Get top 5 bottlenecks
top_5_nodes = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"?? Centrality analysis complete")
print(f"\nTop 5 Bottleneck Intersections:")
for rank, (node_id, cent_val) in enumerate(top_5_nodes, 1):
    x, y = G_proj.nodes[node_id]['x'], G_proj.nodes[node_id]['y']
    print(f"{rank}. Node {node_id}: Centrality = {cent_val:.6f}")
    print(f"   Coordinates: ({x:.2f}, {y:.2f})")

# Store for later use
top_5_node_ids = [node_id for node_id, _ in top_5_nodes]
print(f"\nMost fragile bottleneck: Node {top_5_nodes[0][0]} (centrality: {top_5_nodes[0][1]:.6f})")

In [ ]:
# Step 4: Rainfall-to-Congestion Mapping
print("?? Setting up rainfall-congestion mapping...")

def rain_to_congestion(rainfall_mm, method='threshold'):
    """Map rainfall intensity to congestion factor"""
    if method == 'threshold':
        if rainfall_mm < 10:
            return 0.0  # Normal
        elif rainfall_mm < 40:
            return 0.3  # Slightly slow
        elif rainfall_mm < 80:
            return 0.6  # Severe delay
        else:
            return 0.9  # Almost impassable
    elif method == 'linear':
        return min(rainfall_mm / 100 * 0.9, 0.95)
    elif method == 'exponential':
        return 0.95 * (1 - np.exp(-rainfall_mm/50))
    else:
        return 0.0

# Test the function
rain_test = [0, 10, 40, 80, 100, 130.5]
print("Rainfall-to-Congestion Mapping:")
for rain in rain_test:
    cf = rain_to_congestion(rain, method='threshold')
    print(f"  {rain:6.1f} mm/hr -> cf = {cf:.1f}")

print(f"?? Congestion mapping function ready")

In [ ]:
# Step 5: Dynamic Weighting Application
print("?? Applying dynamic weights based on rainfall...")

def apply_dynamic_weights(G, rainfall_layer, congestion_method='threshold'):
    """Apply dynamic weights to road network edges"""
    G_dyn = G.copy()
    
    for u, v, k, data in G_dyn.edges(data=True, keys=True):
        # Get rainfall for this edge (use node u's rainfall)
        rainfall_mm = rainfall_layer.get(u, 0)
        
        # Calculate congestion factor
        cf = rain_to_congestion(rainfall_mm, method=congestion_method)
        
        # Calculate adjusted travel time
        tt_normal = data.get('travel_time_normal', 60)
        speed_kph = data.get('speed_kph', 40)
        length = data['length']
        
        if cf >= 0.95:  # Almost completely impassable
            data['travel_time_adj'] = float('inf')
        else:
            data['travel_time_adj'] = length / ((speed_kph / 3.6) * (1 - cf))
        
        data['congestion_factor'] = cf
    
    return G_dyn

# Create simulated rainfall layer
np.random.seed(42)
rainfall_values = [0, 5, 15, 25, 50, 65, 90, 130]
rainfall_probs = [0.2, 0.15, 0.15, 0.15, 0.15, 0.1, 0.05, 0.05]

rainfall_layer = {
    node: np.random.choice(rainfall_values, p=rainfall_probs)
    for node in G_proj.nodes()
}

# Apply dynamic weights
G_dyn = apply_dynamic_weights(G_proj, rainfall_layer)

# Statistics
cfs = [d.get('congestion_factor', 0) for _, _, _, d in G_dyn.edges(data=True, keys=True)]
print(f"?? Dynamic weights applied")
print(f"Congestion factor range: {np.min(cfs):.2f} - {np.max(cfs):.2f}")
print(f"Average congestion: {np.mean(cfs):.2f}")
print(f"Edges with cf=0.9 (almost impassable): {sum(1 for cf in cfs if cf >= 0.9)}")

In [ ]:
# Step 6: Isochrone Analysis Functions
print("?? Setting up isochrone analysis...")

def compute_isochrone(G, source_node, weight_attr, time_seconds):
    """Calculate nodes reachable within specified time"""
    distances = nx.single_source_dijkstra_path_length(
        G, source_node, weight=weight_attr, cutoff=time_seconds
    )
    reachable_nodes = set(distances.keys())
    return reachable_nodes, distances

def nodes_to_polygon(G, nodes):
    """Convert reachable nodes to convex hull polygon"""
    if len(nodes) < 3:
        return None, 0.0
    points = [Point(G.nodes[n]['x'], G.nodes[n]['y']) for n in nodes]
    mp = MultiPoint(points)
    polygon = mp.convex_hull
    if polygon.geom_type == 'Polygon':
        return polygon, polygon.area
    return None, 0.0

def get_adaptive_thresholds(G, source_node, weight_attr):
    """Calculate adaptive time thresholds based on network"""
    all_times = dict(nx.single_source_dijkstra_path_length(
        G, source_node, weight=weight_attr
    ))
    max_time = max(all_times.values()) if all_times else 600
    t_short = max_time * 0.35  # 35% of max
    t_long = max_time * 0.65   # 65% of max
    return t_short, t_long

print(f"?? Isochrone analysis functions ready")

In [ ]:
# Step 7: Accessibility Analysis for Critical Facilities
print("?? Analyzing accessibility changes for critical facilities...")

# Use top 3 bottlenecks as critical facilities
selected_facilities = top_5_nodes[:3]
results = []

print(f"Analyzing {len(selected_facilities)} critical facilities:")

for facility_id, facility_cent in selected_facilities:
    print(f"\nFacility {facility_id} (centrality: {facility_cent:.6f}):")
    
    # Pre-disaster analysis
    t_short_b, t_long_b = get_adaptive_thresholds(G_dyn, facility_id, 'travel_time_normal')
    reachable_before_short, _ = compute_isochrone(G_dyn, facility_id, 'travel_time_normal', t_short_b)
    reachable_before_long,  _ = compute_isochrone(G_dyn, facility_id, 'travel_time_normal', t_long_b)
    
    # Post-disaster analysis
    t_short_a, t_long_a = get_adaptive_thresholds(G_dyn, facility_id, 'travel_time_adj')
    reachable_after_short, _ = compute_isochrone(G_dyn, facility_id, 'travel_time_adj', t_short_a)
    reachable_after_long,  _ = compute_isochrone(G_dyn, facility_id, 'travel_time_adj', t_long_a)
    
    # Calculate areas
    _, area_before_short = nodes_to_polygon(G_dyn, reachable_before_short)
    _, area_before_long  = nodes_to_polygon(G_dyn, reachable_before_long)
    _, area_after_short  = nodes_to_polygon(G_dyn, reachable_after_short)
    _, area_after_long   = nodes_to_polygon(G_dyn, reachable_after_long)
    
    # Same-threshold comparison (pre-disaster thresholds for post-disaster)
    reachable_after_same_short, _ = compute_isochrone(G_dyn, facility_id, 'travel_time_adj', t_short_b)
    reachable_after_same_long,  _ = compute_isochrone(G_dyn, facility_id, 'travel_time_adj', t_long_b)
    _, area_after_same_short = nodes_to_polygon(G_dyn, reachable_after_same_short)
    _, area_after_same_long  = nodes_to_polygon(G_dyn, reachable_after_same_long)
    
    # Calculate contraction ratios
    shrink_short = (1 - area_after_same_short / area_before_short) * 100 if area_before_short > 0 else 0
    shrink_long  = (1 - area_after_same_long / area_before_long) * 100 if area_before_long > 0 else 0
    
    result = {
        'facility_id': facility_id,
        'centrality': facility_cent,
        'pre_short_area_km2': area_before_short / 1e6,
        'pre_long_area_km2': area_before_long / 1e6,
        'post_short_area_km2': area_after_short / 1e6,
        'post_long_area_km2': area_after_long / 1e6,
        'same_threshold_short_shrink_%': shrink_short,
        'same_threshold_long_shrink_%': shrink_long,
        'pre_threshold_short_min': t_short_b / 60,
        'pre_threshold_long_min': t_long_b / 60,
        'post_threshold_short_min': t_short_a / 60,
        'post_threshold_long_min': t_long_a / 60,
    }
    
    results.append(result)
    
    print(f"  Pre-disaster: {area_before_short/1e6:.2f} km² ({t_short_b/60:.1f}min), {area_before_long/1e6:.2f} km² ({t_long_b/60:.1f}min)")
    print(f"  Post-disaster: {area_after_short/1e6:.2f} km² ({t_short_a/60:.1f}min), {area_after_long/1e6:.2f} km² ({t_long_a/60:.1f}min)")
    print(f"  Contraction: {shrink_short:.1f}% (short), {shrink_long:.1f}% (long)")

print(f"\n?? Accessibility analysis complete for {len(results)} facilities")

In [ ]:
# Step 8: Create Accessibility Benefit-Cost Table
print("?? Creating accessibility benefit-cost table...")

# Convert results to DataFrame
accessibility_table = pd.DataFrame(results)

# Add priority ranking based on centrality and impact
accessibility_table['priority_score'] = (
    accessibility_table['centrality'] * 0.4 + 
    accessibility_table['same_threshold_short_shrink_%'] * 0.3 +
    accessibility_table['same_threshold_long_shrink_%'] * 0.3
)

accessibility_table = accessibility_table.sort_values('priority_score', ascending=False)
accessibility_table['priority_rank'] = range(1, len(accessibility_table) + 1)

# Display key columns
display_cols = [
    'priority_rank', 'facility_id', 'centrality',
    'same_threshold_short_shrink_%', 'same_threshold_long_shrink_%',
    'priority_score'
]

print("\nAccessibility Benefit-Cost Analysis:")
print(accessibility_table[display_cols].to_string(index=False))

# Save to CSV
accessibility_table.to_csv('accessibility_benefit_cost_table.csv', index=False)
print(f"\n?? Results saved to accessibility_benefit_cost_table.csv")

# Key findings
max_loss_facility = accessibility_table.loc[accessibility_table['same_threshold_short_shrink_%'].idxmax()]
print(f"\nMaximum accessibility loss:")
print(f"  Facility {max_loss_facility['facility_id']}: {max_loss_facility['same_threshold_short_shrink_%']:.1f}% contraction")
print(f"  Centrality: {max_loss_facility['centrality']:.6f}")

In [ ]:
# Step 9: Save Road Network as GraphML
print("?? Saving road network data...")

# Save the projected road network with all attributes
graphml_path = 'hualien_network.graphml'
ox.save_graphml(G_proj, graphml_path)
print(f"?? Road network saved to {graphml_path}")

# Verify the file was created
import os
if os.path.exists(graphml_path):
    file_size = os.path.getsize(graphml_path) / 1024 / 1024  # MB
    print(f"File size: {file_size:.2f} MB")
    print(f"Nodes: {G_proj.number_of_nodes()}")
    print(f"Edges: {G_proj.number_of_edges()}")
    print(f"CRS: {G_proj.graph['crs']}")
else:
    print(f"?? Warning: {graphml_path} not found")

In [ ]:
# Step 10: Visualization - Bottleneck Analysis
print("?? Creating bottleneck visualization...")

fig, ax = plt.subplots(figsize=(15, 12))

# Draw road network
ox.plot_graph(G_proj, ax=ax, node_size=3, node_color='lightgray',
             edge_color='gray', edge_linewidth=0.3, show=False)

# Draw nodes with centrality-based sizing
node_sizes = [centrality[node] * 8000 for node in G_proj.nodes()]
node_colors = [centrality[node] for node in G_proj.nodes()]

for node, size, color in zip(G_proj.nodes(), node_sizes, node_colors):
    x, y = G_proj.nodes[node]['x'], G_proj.nodes[node]['y']
    ax.scatter(x, y, s=size, c=color, cmap='YlOrRd', alpha=0.6, zorder=5)

# Mark top bottlenecks
colors_top5 = ['red', 'orange', 'gold', 'green', 'blue']
for rank, (node_id, cent_val) in enumerate(top_5_nodes):
    x, y = G_proj.nodes[node_id]['x'], G_proj.nodes[node_id]['y']
    ax.plot(x, y, marker='*', markersize=25, color=colors_top5[rank],
           markeredgecolor='black', markeredgewidth=1, zorder=10)
    ax.annotate(f'#{rank+1}\n{cent_val:.4f}', (x, y), fontsize=8, fontweight='bold',
               textcoords='offset points', xytext=(8, 8),
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_title('ARIA v4.0 - Road Network Bottleneck Analysis\n' +
             f'{TARGET_LOCATION} | Node size = Betweenness Centrality',
             fontsize=14, fontweight='bold')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap='YlOrRd', 
                           norm=plt.Normalize(vmin=min(node_colors), vmax=max(node_colors)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6)
cbar.set_label('Betweenness Centrality', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

print(f"?? Visualization complete")

In [ ]:
# Step 11: Final Summary
print("\n" + "="*60)
print("ARIA v4.0 - Hualien Disaster Accessibility Assessment")
print("="*60)

print(f"\n?? Network Analysis Summary:")
print(f"  Target location: {TARGET_LOCATION}")
print(f"  Network nodes: {G_proj.number_of_nodes()}")
print(f"  Network edges: {G_proj.number_of_edges()}")
print(f"  Coordinate system: {G_proj.graph['crs']}")

print(f"\n?? Bottleneck Analysis:")
print(f"  Most fragile bottleneck: Node {top_5_nodes[0][0]}")
print(f"  Centrality value: {top_5_nodes[0][1]:.6f}")
print(f"  Top 5 bottlenecks identified: {[f'Node {nid}' for nid, _ in top_5_nodes]}")

print(f"\n?? Accessibility Impact:")
if len(accessibility_table) > 0:
    avg_short_shrink = accessibility_table['same_threshold_short_shrink_%'].mean()
    avg_long_shrink = accessibility_table['same_threshold_long_shrink_%'].mean()
    max_shrink = accessibility_table['same_threshold_short_shrink_%'].max()
    
    print(f"  Average accessibility loss (short): {avg_short_shrink:.1f}%")
    print(f"  Average accessibility loss (long): {avg_long_shrink:.1f}%")
    print(f"  Maximum accessibility loss: {max_shrink:.1f}%")

print(f"\n?? Rescue Priority Order:")
for i, (_, row) in enumerate(accessibility_table.iterrows()):
    print(f"  {i+1}. Facility {row['facility_id']} (priority: {row['priority_score']:.3f})")

print(f"\n?? Generated Files:")
print(f"  - hualien_network.graphml ({os.path.getsize('hualien_network.graphml')/1024/1024:.1f} MB)")
print(f"  - accessibility_benefit_cost_table.csv")

print(f"\n?? Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)